# 1. Business Understanding

FCN BU todo

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import time
import keras
from keras import layers

import numpy as np
import matplotlib.pyplot as plt

TARGET_REALISTIC = (0.50, 0.55)
TARGET_AMBITIOUS = (0.57, 0.60)

# random seed for repeatability
RANDOM_SEED = 123
keras.utils.set_random_seed(RANDOM_SEED)

# 2. Data Understanding

Todo

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train, y_test = y_train.flatten(), y_test.flatten()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

print("Train: ", x_train.shape, "Test: ", x_test.shape)
print("Pixel range: ", x_train.min(), "-", x_train.max())
print("Class counts: ", np.bincount(y_train))

## Example images

In [ ]:
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].reshape(32, 32, 3))
    plt.title(class_names[y_train[i]])
    plt.axis('off')
plt.show()

# 3. Data Preparation

todo

In [ ]:
x_train = x_train.reshape(50000, 3072).astype("float32") / 255
x_test = x_test.reshape(10000, 3072).astype("float32") / 255

# 4. Modeling

todo

In [ ]:
inputs = keras.Input(shape=(3072,))
x = layers.Dense(1024, activation="relu")(inputs)
x = layers.Dropout(0.3)(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation="softmax")(x)

model = keras.Model(inputs=inputs, outputs=outputs, name="cifar10_fcn")

In [ ]:
model.summary()

In [ ]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(),
    optimizer=keras.optimizers.Adam(),
    metrics=["accuracy"],
)

In [ ]:
t0 = time.time()
history = model.fit(
    x_train, y_train,
    batch_size=64,
    epochs=20,
    validation_split=0.2,
)
print("Training time: ", round(time.time() - t0, 1), " seconds")

# 5. Evaluation

todo

## Learning curves

todo

In [ ]:
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.legend()
plt.grid()
plt.show()

## Test accuracy

In [ ]:
test_scores = model.evaluate(x_test, y_test, verbose=2)

print("Test loss: ", test_scores[0])
print("Test accuracy: ", test_scores[1])

## Probability distributions

todo

In [ ]:
y = model(x_test)
start = 2000

for k in range(10):
    plt.figure(figsize=(8, 2))
    plt.subplot(1, 2, 1)
    plt.imshow(x_test[start+k].reshape(32, 32, 3))
    plt.title("True: " + class_names[y_test[start+k]])
    plt.subplot(1, 2, 2)
    plt.bar(np.arange(10), y[start+k].numpy().flatten())
    plt.xticks(range(10), class_names, rotation=45)

## Misclassified images

todo

In [ ]:
y_pred = np.argmax(model.predict(x_test), axis=1)
wrong = np.where(y_pred != y_test)[0]

print("Missclassified: ", len(wrong), "/", len(y_test))

for k in range(10):
    i = wrong[k]
    plt.subplot(2, 5, k+1)
    plt.imshow(x_test[i].reshape(32, 32, 3))
    plt.title(f"True: {class_names[y_test[i]]}\nPredicted: {class_names[y_pred[i]]}", fontsize=8)
    plt.axis("off")
plt.show()